In [ ]:
import numpy as np
import xarray as xr
import uxarray as ux
import requests
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib import colors
from matplotlib.collections import PolyCollection

import cartopy.crs as ccrs

# ============================================================
# settings
# ============================================================
VAR_NAME = "ts"          # "ts" or "pr"
TIME_INDEX = 0
ZOOM = 9                  # 4/5/6 good for actual HEALPix polygons; high zoom falls back
NX = 720
NY = 360
MAX_DEST_POLYGONS = 200_000

DATASET_ID = "eerie-future-ssp245-v20240618"
STAC_ITEM_URL = (
    "https://stac2.cloud.dkrz.de/fastapi/collections/"
    "eerie-eerie-mpi-m-icon-esm-er-highres-future-ssp245-v20240618/items/"
    "eerie-eerie-mpi-m-icon-esm-er-highres-future-ssp245-v20240618-"
    "disk.model-output.icon-esm-er.highres-future-ssp245.v20240618.atmos.native.2d_monthly_mean-zarr-kerchunk"
)
ASSET = "dkrz-disk"

GRIDFILE = "/pool/data/ICON/grids/public/mpim/0033/icon_grid_0033_R02B08_G.nc"
OLD_ROOT = "/scratch/k/k202181/remap_eerie/eerie-future-ssp245-v20240618.zarr"
GD_ROOT = "/scratch/k/k202181/remap_eerie/gd-eerie-future-ssp245-v20240618.zarr"

# ============================================================
# helpers
# ============================================================
def to_deg(arr):
    arr = np.asarray(arr, dtype=np.float64)
    if np.nanmax(np.abs(arr)) <= np.pi + 0.01:
        return np.rad2deg(arr)
    return arr


def nanstats(arr):
    arr = np.asarray(arr, dtype=np.float64)
    return {
        "min": float(np.nanmin(arr)),
        "max": float(np.nanmax(arr)),
        "mean": float(np.nanmean(arr)),
    }


def decode_time_like(da: xr.DataArray):
    vals = np.asarray(da.values)
    if np.issubdtype(vals.dtype, np.datetime64):
        return vals.astype("datetime64[ns]")
    try:
        tmp = xr.decode_cf(xr.Dataset(coords={"time": da}))
        vals2 = np.asarray(tmp["time"].values)
        if np.issubdtype(vals2.dtype, np.datetime64):
            return vals2.astype("datetime64[ns]")
        return vals2
    except Exception:
        return vals


def rasterize_face_centers(lon_deg, lat_deg, values, nx=720, ny=360):
    lon = ((np.asarray(lon_deg, dtype=np.float64) + 180.0) % 360.0) - 180.0
    lat = np.clip(np.asarray(lat_deg, dtype=np.float64), -90.0, 90.0)
    val = np.asarray(values, dtype=np.float64)

    good = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(val)
    lon = lon[good]
    lat = lat[good]
    val = val[good]

    ix = np.floor((lon + 180.0) / 360.0 * nx).astype(np.int64)
    iy = np.floor((lat + 90.0) / 180.0 * ny).astype(np.int64)

    ix = np.clip(ix, 0, nx - 1)
    iy = np.clip(iy, 0, ny - 1)

    flat = iy * nx + ix
    sums = np.bincount(flat, weights=val, minlength=nx * ny)
    counts = np.bincount(flat, minlength=nx * ny)

    out = np.full(nx * ny, np.nan, dtype=np.float64)
    m = counts > 0
    out[m] = sums[m] / counts[m]
    return out.reshape(ny, nx)


def build_face_polygons(grid, values):
    """
    Build actual cell polygons for a UXarray grid.

    Longitudes are adjusted around each face center to avoid dateline-spanning artifacts.
    """
    node_lon = to_deg(grid.node_lon.values)
    node_lat = to_deg(grid.node_lat.values)
    face_lon = to_deg(grid.face_lon.values)

    conn = np.asarray(grid.face_node_connectivity.values)
    n_nodes_per_face = np.asarray(grid.n_nodes_per_face.values, dtype=np.int64)
    vals = np.asarray(values, dtype=np.float64)

    polys = []
    poly_vals = []

    n_face = int(n_nodes_per_face.size)
    for i in range(n_face):
        if not np.isfinite(vals[i]):
            continue

        m = int(n_nodes_per_face[i])
        ids = np.asarray(conn[i, :m], dtype=np.int64)
        ids = ids[ids >= 0]

        if ids.size < 3:
            continue

        lons = node_lon[ids].astype(np.float64)
        lats = node_lat[ids].astype(np.float64)
        clon = float(face_lon[i])

        # adjust node longitudes to stay close to face center
        lons = ((lons - clon + 180.0) % 360.0) - 180.0 + clon

        polys.append(np.column_stack([lons, lats]))
        poly_vals.append(vals[i])

    return polys, np.asarray(poly_vals, dtype=np.float64)


def print_time_check(label, source_time_all, other_time_all):
    same_len = len(source_time_all) == len(other_time_all)
    same_values = same_len and np.array_equal(source_time_all, other_time_all)

    print(label)
    print(f"  time axis same length : {same_len}")
    print(f"  time axis exact match : {same_values}")
    print(f"  source time[{TIME_INDEX}] : {source_time_all[TIME_INDEX]}")
    print(f"  other  time[{TIME_INDEX}] : {other_time_all[TIME_INDEX]}")

    if not same_values and same_len:
        diff_idx = np.flatnonzero(source_time_all != other_time_all)
        if diff_idx.size > 0:
            j = int(diff_idx[0])
            print(f"  first differing time index: {j}")
            print(f"    source[{j}] = {source_time_all[j]}")
            print(f"    other [{j}] = {other_time_all[j]}")
    print()


def plot_healpix_panel(ax, grid, lon_deg, lat_deg, values, title, vmin, vmax, nx, ny, max_polygons):
    n_face = int(grid.n_face)
    if n_face <= max_polygons:
        polys, poly_vals = build_face_polygons(grid, values)
        coll = PolyCollection(
            polys,
            array=poly_vals,
            edgecolors="none",
            closed=True,
            transform=ccrs.PlateCarree(),
            cmap=plt.get_cmap(),
            norm=colors.Normalize(vmin=vmin, vmax=vmax),
        )
        ax.add_collection(coll)
        ax.coastlines(linewidth=0.5)
        ax.set_global()
        ax.set_title(title)
        return "actual HEALPix polygons"

    img = rasterize_face_centers(lon_deg, lat_deg, values, nx=nx, ny=ny)
    ax.imshow(
        img,
        origin="lower",
        extent=(-180, 180, -90, 90),
        transform=ccrs.PlateCarree(),
        vmin=vmin,
        vmax=vmax,
    )
    ax.coastlines(linewidth=0.5)
    ax.set_global()
    ax.set_title(title)
    return "rasterized fallback"


# ============================================================
# open source dataset
# ============================================================
resp = requests.get(STAC_ITEM_URL)
resp.raise_for_status()
item = resp.json()

source_ds = xr.open_dataset(
    item["assets"][ASSET]["href"],
    **item["assets"][ASSET]["xarray:open_kwargs"],
    storage_options=item["assets"][ASSET].get("xarray:storage_options"),
)

# ============================================================
# open source grid + source field
# ============================================================
src_grid = ux.open_grid(GRIDFILE, chunks=-1)

src_da = source_ds[VAR_NAME].isel(time=TIME_INDEX).squeeze(drop=True).compute()
src_vals = np.asarray(src_da.values)
src_lon = to_deg(src_grid.face_lon.values)
src_lat = to_deg(src_grid.face_lat.values)

# ============================================================
# open old and grid-doctor outputs
# ============================================================
old_store_path = Path(OLD_ROOT) / "multiscales" / f"zoom_{ZOOM}"
gd_store_path = Path(GD_ROOT) / "multiscales" / f"zoom_{ZOOM}"

old_ds = xr.open_zarr(old_store_path, consolidated=False)
gd_ds = xr.open_zarr(gd_store_path, consolidated=False)

old_da = old_ds[VAR_NAME].isel(time=TIME_INDEX).squeeze(drop=True)
gd_da = gd_ds[VAR_NAME].isel(time=TIME_INDEX).squeeze(drop=True)

old_vals = np.asarray(old_da.values)
gd_vals = np.asarray(gd_da.values)

dst_grid = ux.Grid.from_healpix(zoom=ZOOM, pixels_only=False, nest=True)
dst_lon = to_deg(dst_grid.face_lon.values)
dst_lat = to_deg(dst_grid.face_lat.values)

# ============================================================
# stats
# ============================================================
src_stats = nanstats(src_vals)
old_stats = nanstats(old_vals)
gd_stats = nanstats(gd_vals)

print(f"dataset    : {DATASET_ID}")
print(f"variable   : {VAR_NAME}")
print(f"time index : {TIME_INDEX}")
print(f"zoom       : {ZOOM}")
print(f"old store  : {old_store_path}")
print(f"gd store   : {gd_store_path}")
print()
print(
    f"SOURCE -> min={src_stats['min']:.10g}, max={src_stats['max']:.10g}, mean={src_stats['mean']:.10g}"
)
print(
    f"OLD    -> min={old_stats['min']:.10g}, max={old_stats['max']:.10g}, mean={old_stats['mean']:.10g}"
)
print(
    f"GD     -> min={gd_stats['min']:.10g}, max={gd_stats['max']:.10g}, mean={gd_stats['mean']:.10g}"
)
print()

# ============================================================
# time checks
# ============================================================
src_time_all = decode_time_like(source_ds["time"])
old_time_all = decode_time_like(old_ds["time"])
gd_time_all = decode_time_like(gd_ds["time"])

print("TIME CHECK")
print(f"source time dtype : {source_ds['time'].dtype}")
print(f"old    time dtype : {old_ds['time'].dtype}")
print(f"gd     time dtype : {gd_ds['time'].dtype}")
print()
print_time_check("SOURCE vs OLD", src_time_all, old_time_all)
print_time_check("SOURCE vs GD", src_time_all, gd_time_all)

# ============================================================
# common color scale
# ============================================================
finite_src = src_vals[np.isfinite(src_vals)]
finite_old = old_vals[np.isfinite(old_vals)]
finite_gd = gd_vals[np.isfinite(gd_vals)]
all_vals = np.concatenate([finite_src, finite_old, finite_gd])

vmin = np.nanpercentile(all_vals, 1)
vmax = np.nanpercentile(all_vals, 99)
if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
    vmin = np.nanmin(all_vals)
    vmax = np.nanmax(all_vals)

# ============================================================
# plotting
# ============================================================
fig = plt.figure(figsize=(24, 6), constrained_layout=True)
proj = ccrs.Robinson()

ax1 = fig.add_subplot(1, 3, 1, projection=proj)
ax2 = fig.add_subplot(1, 3, 2, projection=proj)
ax3 = fig.add_subplot(1, 3, 3, projection=proj)

src_img = rasterize_face_centers(src_lon, src_lat, src_vals, nx=NX, ny=NY)
ax1.imshow(
    src_img,
    origin="lower",
    extent=(-180, 180, -90, 90),
    transform=ccrs.PlateCarree(),
    vmin=vmin,
    vmax=vmax,
)
ax1.coastlines(linewidth=0.5)
ax1.set_global()
ax1.set_title(f"Native rasterized: {VAR_NAME}")

old_mode = plot_healpix_panel(
    ax2,
    dst_grid,
    dst_lon,
    dst_lat,
    old_vals,
    f"Old remap: {VAR_NAME}, zoom={ZOOM}",
    vmin,
    vmax,
    NX,
    NY,
    MAX_DEST_POLYGONS,
)

gd_mode = plot_healpix_panel(
    ax3,
    dst_grid,
    dst_lon,
    dst_lat,
    gd_vals,
    f"Grid-doctor remap: {VAR_NAME}, zoom={ZOOM}",
    vmin,
    vmax,
    NX,
    NY,
    MAX_DEST_POLYGONS,
)

sm = plt.cm.ScalarMappable(norm=colors.Normalize(vmin=vmin, vmax=vmax), cmap=plt.get_cmap())
sm.set_array([])
fig.colorbar(sm, ax=[ax1, ax2, ax3], shrink=0.85)

print(f"old plot mode: {old_mode}")
print(f"gd  plot mode: {gd_mode}")

plt.show()


In [ ]:
import numpy as np
import xarray as xr
from pathlib import Path

ZOOM = globals().get("ZOOM", 6)
OLD_ROOT = globals().get("OLD_ROOT", "/scratch/k/k202181/remap_eerie/eerie-future-ssp245-v20240618.zarr")
GD_ROOT = globals().get("GD_ROOT", "/scratch/k/k202181/remap_eerie/gd-eerie-future-ssp245-v20240618.zarr")

old_ds = xr.open_zarr(Path(OLD_ROOT) / "multiscales" / f"zoom_{ZOOM}", consolidated=False)
gd_ds = xr.open_zarr(Path(GD_ROOT) / "multiscales" / f"zoom_{ZOOM}", consolidated=False)


def compare_arrays(a, b):
    a = np.asarray(a)
    b = np.asarray(b)

    if a.shape != b.shape:
        return False, f"shape differs: {a.shape} vs {b.shape}"
    if a.dtype != b.dtype:
        return False, f"dtype differs: {a.dtype} vs {b.dtype}"

    if np.issubdtype(a.dtype, np.floating):
        exact = np.array_equal(a, b, equal_nan=True)
        diff = np.abs(a - b)
        return exact, (
            f"exact_equal={exact}, max_abs_diff={np.nanmax(diff):.10g}, "
            f"mean_abs_diff={np.nanmean(diff):.10g}"
        )

    if np.issubdtype(a.dtype, np.datetime64):
        exact = np.array_equal(a, b)
        return exact, f"exact_equal={exact}"

    exact = np.array_equal(a, b)
    return exact, f"exact_equal={exact}"


vars_old = set(old_ds.variables)
vars_gd = set(gd_ds.variables)
common = sorted(vars_old & vars_gd)

print(f"zoom        : {ZOOM}")
print("same names  :", vars_old == vars_gd)
if vars_old != vars_gd:
    print("only in old :", sorted(vars_old - vars_gd))
    print("only in gd  :", sorted(vars_gd - vars_old))
print()

all_ok = vars_old == vars_gd
for name in common:
    ok, msg = compare_arrays(old_ds[name].values, gd_ds[name].values)
    print(f"{name}: {msg}")
    if not ok:
        all_ok = False

print()
print("FINAL:", "all common arrays identical" if all_ok else "differences found")


In [ ]:
from pathlib import Path
import xarray as xr

OLD_ROOT = Path(globals().get("OLD_ROOT", "/scratch/k/k202181/remap_eerie/eerie-future-ssp245-v20240618.zarr"))
GD_ROOT = Path(globals().get("GD_ROOT", "/scratch/k/k202181/remap_eerie/gd-eerie-future-ssp245-v20240618.zarr"))


def show_store(label, root, sample_zooms=(0, 6, 9)):
    print()
    print(f"{label}: {root}")
    multiscales_root = root / "multiscales"
    zooms = sorted(p.name for p in multiscales_root.iterdir() if p.name.startswith("zoom_"))
    print("available zooms:", zooms)

    for zoom in sample_zooms:
        ds = xr.open_zarr(multiscales_root / f"zoom_{zoom}", consolidated=False)
        print(f"zoom_{zoom}: vars={list(ds.data_vars)} sizes={dict(ds.sizes)}")


show_store("OLD REMAP", OLD_ROOT)
show_store("GRID-DOCTOR REMAP", GD_ROOT)
